# 02. Grammar and core contracts

This component holds the three contracts the search is built on.

* `Grammar` (in `alpha_rule.grammar`) is the single source of truth for the search
  space. It defines the start state, which productions are legal at a state, how a
  production transforms a state, and when a state is terminal. Swap the grammar and
  you change what the search explores, without touching the MCTS or the network.
* `MCTSRuleNode` (in `alpha_rule.mcts.node`) is the search state. It is pure data:
  statistics, tree links, and flags. It carries no Allen logic and no behaviour.
* `Evaluator` (in `alpha_rule.evaluation.evaluator`) is the contract for scoring a
  state. Anything with an `evaluate(node) -> EvalResult` method is an evaluator.

`AllenIntervalGrammar` is the concrete grammar for Allen interval rules, built on
the symbolic core from component 01.

In [1]:
import alpha_rule
print("alpha_rule", alpha_rule.__version__)

alpha_rule 0.1.0


## Elements

`alpha_rule.grammar.production`
* `Production(name, kind)`: one grammar production. `name` is the token the network
  uses. `kind` is a free-form label the grammar uses internally to tell its own
  productions apart; the Allen grammar uses `"event"`, `"relation"`, and
  `"terminal"`, but any non-empty string works. The MCTS and the network never read
  `kind`, so a new grammar reuses `Production` unchanged.

`alpha_rule.grammar.grammar`
* `Grammar` protocol: `root()`, `vocab()`, `applicable_productions(state)`,
  `apply(state, production)`, `is_terminal(state)`.

`alpha_rule.grammar.allen`
* `AllenIntervalGrammar(event_types, relations)`: the concrete grammar.
* `should_add_event(level)`: whether construction step `level` adds an event type
  or an Allen relation (the triangular schedule).

`alpha_rule.mcts.node`
* `MCTSRuleNode`: the pure-state node. Holds `name`, `level`, `parent`, `children`,
  flags, and the MCTS statistics. `is_fully_expanded()` compares `len(children)`
  against `n_possible_actions`, which the grammar stamps on each node it builds.

`alpha_rule.evaluation.evaluator`
* `Evaluator` protocol: `evaluate(node) -> EvalResult`.
* `EvalResult(value, priors=None)`: the scalar score, plus optional action priors.
* `RuleStringNode(name)`: a tiny node-like object exposing only `.name`, for when
  you want to score a bare rule string.

### Productions

A `Production` is one legal move: a token to append plus a `kind` label. The `kind`
is whatever the grammar finds useful inside its own `apply`; the core never inspects
it. So `Production` is generic, and you reuse it for any grammar without changes.
The only rules are that `name` and `kind` are non-empty.

In [2]:
from alpha_rule.grammar.production import Production

print(Production(name="A", kind="event"))
print(Production(name="END_RULE", kind="terminal"))
# a different grammar can pick its own labels:
print(Production(name="digit", kind="token"))

# name and kind must both be non-empty:
for bad in (dict(name="", kind="event"), dict(name="A", kind="")):
    try:
        Production(**bad)
    except ValueError as e:
        print("rejected:", e)

Production(name='A', kind='event')
Production(name='END_RULE', kind='terminal')
Production(name='digit', kind='token')
rejected: Production.name cannot be empty
rejected: Production.kind cannot be empty


### The grammar

In [3]:
from alpha_rule.grammar.allen import AllenIntervalGrammar, should_add_event

g = AllenIntervalGrammar(event_types=("A", "B", "C"))
print("vocab:", g.vocab())
root = g.root()
print("root :", root.name, "| applicable:", [p.name for p in g.applicable_productions(root)])

vocab: ['A', 'B', 'C', '<', 'm', 'o', 's', 'd', 'f', '=', '>', 'mi', 'oi', 'si', 'di', 'fi', 'END_RULE']
root : <ROOT> | applicable: ['A', 'B', 'C']


### Building a rule, level by level

The grammar alternates event steps and relation steps following the triangular
schedule (`should_add_event`). At every non-root state `END_RULE` is also offered.

In [4]:
node = g.root()
for _ in range(3):
    prods = g.applicable_productions(node)
    kind = "event" if should_add_event(node.level) else "relation"
    print(f"level {node.level}: {kind:8} step  applicable = {[p.name for p in prods]}")
    nxt = next(p for p in prods if p.kind != "terminal")   # go one level deeper
    node = g.apply(node, nxt)
print("built:", repr(node.name))

level 0: event    step  applicable = ['A', 'B', 'C']
level 1: event    step  applicable = ['END_RULE', 'A', 'B', 'C']
level 2: relation step  applicable = ['END_RULE', '<', 'm', 'o', 's', 'd', 'f', '=', '>', 'mi', 'oi', 'si', 'di', 'fi']
built: 'A A <'


### Finishing a rule

`END_RULE` produces a terminal state with nothing after it.

In [5]:
a = g.apply(g.root(), g.applicable_productions(g.root())[0])   # one event
end = next(p for p in g.applicable_productions(a) if p.name == "END_RULE")
terminal = g.apply(a, end)
print("terminal     :", repr(terminal.name))
print("is_terminal  :", g.is_terminal(terminal))
print("applicable   :", g.applicable_productions(terminal))
print("child linked :", terminal.parent is a and terminal in a.children)

terminal     : 'A <END>'
is_terminal  : True
applicable   : []
child linked : True


### The node is pure state

`MCTSRuleNode` is data only. `is_fully_expanded` is driven entirely by
`n_possible_actions` versus the number of children.

In [6]:
from alpha_rule.mcts.node import MCTSRuleNode

n = MCTSRuleNode(name="<ROOT>", n_possible_actions=2)
print("name:", n.name, "| level:", n.level, "| N:", n.N, "| Q_max:", n.Q_max, "| prior:", n.prior)
print("fully expanded with 0 of 2 children:", n.is_fully_expanded())
n.children.append(MCTSRuleNode(name="A", parent=n, parent_action="A"))
n.children.append(MCTSRuleNode(name="B", parent=n, parent_action="B"))
print("fully expanded with 2 of 2 children:", n.is_fully_expanded())

name: <ROOT> | level: 0 | N: 0 | Q_max: -inf | prior: 1.0
fully expanded with 0 of 2 children: False
fully expanded with 2 of 2 children: True


### The evaluator contract

Any object with an `evaluate(node) -> EvalResult` method is an `Evaluator`. It
reads only `node.name`, so a `RuleStringNode` is enough to score a bare string.

In [7]:
from alpha_rule.evaluation.evaluator import Evaluator, EvalResult, RuleStringNode

class LengthEvaluator:
    def evaluate(self, node):
        return EvalResult(value=float(len(node.name.split())))

ev = LengthEvaluator()
print("is an Evaluator:", isinstance(ev, Evaluator))           # runtime checkable protocol
print("score of 'A B <':", ev.evaluate(RuleStringNode(name="A B <")).value)

is an Evaluator: True
score of 'A B <': 3.0


### Swapping the grammar

The grammar is the only thing that knows about Allen intervals. Here is a different
grammar over binary strings that implements the same protocol, built by hand. It
picks its own `kind` labels (`"token"`, `"stop"`), since the core never reads them.
The node and the productions carry no Allen assumptions. The proof that such a
grammar also drives the full search lives in component 03.

In [8]:
from alpha_rule.grammar.production import Production

class ToyGrammar:
    TOKENS = ("0", "1")
    MAX_LEN = 3

    def _new(self, name, level, parent=None, parent_action=None, is_terminal=False):
        node = MCTSRuleNode(name=name, level=level, parent=parent,
                            parent_action=parent_action, is_terminal=is_terminal)
        node.n_possible_actions = len(self.applicable_productions(node))
        return node

    def root(self):
        return self._new("<ROOT>", 0)

    def vocab(self):
        return list(self.TOKENS) + ["END"]

    def applicable_productions(self, state):
        if state.is_terminal or state.level >= self.MAX_LEN:
            return []
        prods = [Production(name=t, kind="token") for t in self.TOKENS]
        if state.name != "<ROOT>":
            prods = [Production(name="END", kind="stop")] + prods
        return prods

    def apply(self, state, production):
        is_term = production.kind == "stop"
        base = "" if state.name == "<ROOT>" else state.name
        name = (base + " <END>").strip() if is_term else (base + " " + production.name).strip()
        child = self._new(name, state.level + 1, parent=state,
                          parent_action=production.name, is_terminal=is_term)
        state.children.append(child)
        return child

    def is_terminal(self, state):
        return state.is_terminal

toy = ToyGrammar()
node = toy.root()
node = toy.apply(node, toy.applicable_productions(node)[0])    # 0
node = toy.apply(node, toy.applicable_productions(node)[1])    # then 1
print("toy vocab :", toy.vocab())
print("built     :", repr(node.name))
print("only toy tokens used:", set(node.name.split()) <= set(toy.vocab()) | {"<END>"})

toy vocab : ['0', '1', 'END']
built     : '0 0'
only toy tokens used: True


## Basic checks

A few asserts that mirror the unit tests. Run the cell. If it prints `ok`, the
contracts behave as expected.

In [9]:
# Grammar: root offers only events; non-root prepends END_RULE.
g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
assert {"A", "B", "<", ">", "END_RULE"} <= set(g.vocab())
root = g.root()
assert [p.name for p in g.applicable_productions(root)] == ["A", "B"]
assert root.n_possible_actions == 2
child = g.apply(root, g.applicable_productions(root)[0])
assert g.applicable_productions(child)[0].name == "END_RULE"
assert child.parent is root and child in root.children

# apply END_RULE gives a terminal with no productions after it.
end = next(p for p in g.applicable_productions(child) if p.name == "END_RULE")
t = g.apply(child, end)
assert t.is_terminal and t.name.endswith("<END>") and g.applicable_productions(t) == []

# A production of the wrong kind for the step is rejected.
relation_prod = Production(name="<", kind="relation")
try:
    g.apply(g.root(), relation_prod)            # root is an event step
    raise AssertionError("expected ValueError")
except ValueError as e:
    assert "expects a production of kind" in str(e)

# Node: pure state, no behaviour methods.
n = MCTSRuleNode(name="<ROOT>")
assert n.N == 0 and n.prior == 1.0 and n.is_dead is False
for removed in ("expand", "ucb_score", "backpropagate", "node_value"):
    assert not hasattr(n, removed)

# RuleStringNode: frozen, hashable, readable by any evaluator.
rs = RuleStringNode(name="A B <")
assert rs.name == "A B <" and hash(rs) == hash(RuleStringNode(name="A B <"))
assert LengthEvaluator().evaluate(rs).value == 3.0

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_grammar
python -m alpha_rule.tests.run_tests test_mcts_node
python -m alpha_rule.tests.run_tests test_rule_string_node
```

`test_grammar` includes a swap test that drives the full search with a custom
grammar. It skips here because it needs `run_self_play`, which arrives in
component 03; it runs automatically once that lands.